# Agent 1 — Baseline Computation

Reads historical clean data from both pilot tables and computes
statistical baselines for each configured check type.

**Run this notebook before running the validation agent.**
**It must be re-run whenever new historical data is available.**

Reads from:
  - uc1_table_registry_b
  - uc1_column_registry_b
  - uc1_column_validation_registry_b
  - uc1_table_validation_registry_b
  - The actual pilot tables (raw transactions and inventory)

Writes to:
  - uc1_baseline_metrics_b

In [0]:
import uuid
import json
from datetime import date, datetime, timedelta
from dateutil.relativedelta import relativedelta
from pyspark.sql import functions as F
from pyspark.sql import DataFrame
from pyspark.sql.types import (
    StructType, StructField,
    StringType, DateType, TimestampType,
    DoubleType, IntegerType, BooleanType
)
import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# -- Catalog and Schema -----------------------------------------------------

dbutils.widgets.text("catalog", "learning_dev", "Target Catalog")
dbutils.widgets.text("schema", "s819534", "Target Schema")
dbutils.widgets.dropdown("run_mode", "SCHEDULED", ["SCHEDULED", "INITIAL", "CUSTOM", "BACKFILL", "VALIDATE_ONLY"], "Run Mode")
dbutils.widgets.text("table_filter", "", "Table Filter (comma-separated FQNs or table names)")
dbutils.widgets.text("as_of_date", "", "As Of Date (YYYY-MM-DD)")
dbutils.widgets.text("baseline_start", "", "Baseline Start (YYYY-MM-DD)")
dbutils.widgets.text("baseline_end", "", "Baseline End (YYYY-MM-DD)")
dbutils.widgets.text("window_days", "30", "Window Days")
dbutils.widgets.dropdown("window_policy", "MONTH", ["MONTH", "DAYS"], "Window Policy")
dbutils.widgets.text("refresh_days", "1,15", "Refresh Days (comma-separated)")
dbutils.widgets.text("job_run_id", "", "Job Run ID (optional)")

CATALOG = dbutils.widgets.get("catalog")
SCHEMA  = dbutils.widgets.get("schema")
RUN_MODE = (dbutils.widgets.get("run_mode") or "SCHEDULED").strip().upper()
TABLE_FILTER = (dbutils.widgets.get("table_filter") or "").strip()
AS_OF_DATE_RAW = (dbutils.widgets.get("as_of_date") or "").strip()
BASELINE_START_RAW = (dbutils.widgets.get("baseline_start") or "").strip()
BASELINE_END_RAW = (dbutils.widgets.get("baseline_end") or "").strip()
WINDOW_DAYS_RAW = (dbutils.widgets.get("window_days") or "30").strip()
WINDOW_POLICY = (dbutils.widgets.get("window_policy") or "MONTH").strip().upper()
REFRESH_DAYS_RAW = (dbutils.widgets.get("refresh_days") or "1,15").strip()
JOB_RUN_ID = (dbutils.widgets.get("job_run_id") or "").strip()

# -- Metadata Table References ----------------------------------------------

TABLE_REGISTRY              = f"{CATALOG}.{SCHEMA}.uc1_table_registry_b"
COLUMN_REGISTRY             = f"{CATALOG}.{SCHEMA}.uc1_column_registry_b"
COLUMN_VALIDATION_REGISTRY  = f"{CATALOG}.{SCHEMA}.uc1_column_validation_registry_b"
TABLE_VALIDATION_REGISTRY   = f"{CATALOG}.{SCHEMA}.uc1_table_validation_registry_b"
BASELINE_METRICS            = f"{CATALOG}.{SCHEMA}.uc1_baseline_metrics_b"

ALLOWED_METADATA_PREFIX = "learning_dev.s819534."
def assert_metadata_target(table_name: str):
    if not (table_name or "").startswith(ALLOWED_METADATA_PREFIX):
        raise ValueError(
            f"Blocked write target '{table_name}'. Allowed prefix: {ALLOWED_METADATA_PREFIX}"
        )

# -- Baseline Settings ------------------------------------------------------

MIN_BASELINE_DAYS    = 15
MIN_SAMPLE_COUNT     = 15
BASELINE_DATE        = date.today()
COMPUTED_BY          = "05_baseline_computation"
BASELINE_VERSION_ID  = str(uuid.uuid4())

WINDOW_START_DATE = None
WINDOW_END_DATE = None
EFFECTIVE_DATE = None
BASELINE_WINDOW_DAYS = None

WINDOW_POLICY_MONTH = "MONTH"
WINDOW_POLICY_DAYS = "DAYS"

TABLE_CHECK_NAME_MAP = {
    "SCHEMA_DRIFT": "SCHEMA_CONFORMANCE",
    "DUPLICATE_CHECK": "DUPLICATE_INGESTION",
}

def _as_int_or_default(raw, default: int) -> int:
    try:
        return int(raw)
    except Exception:
        return default

def normalize_lag_days(lag_days) -> int:
    return max(1, _as_int_or_default(lag_days, 1))

def compute_default_window(as_of_date: date, lag_days, policy: str = WINDOW_POLICY_MONTH, window_days: int = 30):
    """
    Default window rule:
      end = as_of_date - lag_days
      MONTH: start = (end + 1 day) - 1 month
      DAYS : start = end - (window_days - 1)
    """
    lag = normalize_lag_days(lag_days)
    end_dt = as_of_date - timedelta(days=lag)

    if (policy or WINDOW_POLICY_MONTH).upper() == WINDOW_POLICY_DAYS:
        wd = max(1, _as_int_or_default(window_days, 30))
        start_dt = end_dt - timedelta(days=wd - 1)
    else:
        start_dt = (end_dt + timedelta(days=1)) - relativedelta(months=1)

    return start_dt, end_dt

def _safe_parse_date(raw: str):
    raw = (raw or "").strip()
    if not raw:
        return None
    return datetime.fromisoformat(raw[:10]).date()

def _parse_refresh_days(raw: str):
    vals = []
    for token in (raw or "").split(","):
        t = token.strip()
        if not t:
            continue
        try:
            day = int(t)
        except Exception:
            continue
        if 1 <= day <= 31:
            vals.append(day)
    return sorted(set(vals)) or [1, 15]

ANCHOR_DATE = _safe_parse_date(AS_OF_DATE_RAW) or BASELINE_DATE
REFRESH_DAYS = _parse_refresh_days(REFRESH_DAYS_RAW)

# SCHEDULED runs compute only on refresh days. Other modes always compute.
if RUN_MODE == "SCHEDULED" and ANCHOR_DATE.day not in REFRESH_DAYS:
    dbutils.notebook.exit("SKIPPED_NOT_REFRESH_DAY")

# Self-checks for month-back policy parity with onboarding helper.
_self_checks = [
    (date(2026, 11, 1), 1, "2026-10-01", "2026-10-31"),
    (date(2026, 11, 15), 1, "2026-10-15", "2026-11-14"),
    (date(2026, 3, 1), 1, "2026-02-01", "2026-02-28"),
    (date(2026, 3, 31), 1, "2026-02-28", "2026-03-30"),
    (date(2026, 11, 15), 14, "2026-10-02", "2026-11-01"),
]
for _as_of, _lag, _exp_start, _exp_end in _self_checks:
    _s, _e = compute_default_window(_as_of, _lag, WINDOW_POLICY_MONTH)
    print(f"SELF_CHECK as_of={_as_of} lag={_lag} -> {_s}..{_e} | expected {_exp_start}..{_exp_end}")

# -- Baseline Schema --------------------------------------------------------

BASELINE_SCHEMA = StructType([
    StructField("baseline_id",          StringType(),  False),
    StructField("baseline_version_id",  StringType(),  False),
    StructField("fully_qualified_name", StringType(),  False),
    StructField("column_name",          StringType(),  True),
    StructField("check_type",           StringType(),  False),
    StructField("metric_name",          StringType(),  False),
    StructField("baseline_date",        DateType(),    False),
    StructField("effective_date",       DateType(),    False),
    StructField("window_start",         DateType(),    False),
    StructField("window_end",           DateType(),    False),
    StructField("run_mode",             StringType(),  False),
    StructField("job_run_id",           StringType(),  True),
    StructField("baseline_window_days", IntegerType(), False),
    StructField("expected_value",       DoubleType(),  True),
    StructField("mean_value",           DoubleType(),  True),
    StructField("stddev_value",         DoubleType(),  True),
    StructField("min_value",            DoubleType(),  True),
    StructField("max_value",            DoubleType(),  True),
    StructField("p25_value",            DoubleType(),  True),
    StructField("p50_value",            DoubleType(),  True),
    StructField("p75_value",            DoubleType(),  True),
    StructField("p95_value",            DoubleType(),  True),
    StructField("sample_count",         IntegerType(), False),
    StructField("is_sufficient_data",   BooleanType(), False),
    StructField("baseline_status",      StringType(),  False),
    StructField("baseline_detail",      StringType(),  True),
    StructField("computed_timestamp",   TimestampType(), False),
    StructField("computed_by",          StringType(),  False),
])

print("=" * 60)
print("AGENT 1 — BASELINE COMPUTATION")
print("=" * 60)
print(f"Catalog             : {CATALOG}")
print(f"Schema              : {SCHEMA}")
print(f"Run mode            : {RUN_MODE}")
print(f"Table filter        : {TABLE_FILTER or 'All active tables'}")
print(f"As-of date          : {ANCHOR_DATE}")
print(f"Window policy       : {WINDOW_POLICY}")
print(f"Refresh days        : {REFRESH_DAYS}")
print(f"Baseline version ID : {BASELINE_VERSION_ID}")
print(f"Min baseline days   : {MIN_BASELINE_DAYS}")
print(f"Job run id          : {JOB_RUN_ID or 'N/A'}")
print(f"Computed by         : {COMPUTED_BY}")
print("=" * 60)

## Step 1: Load Active Configuration from Metadata Tables

In [0]:
def load_active_tables(table_filter: str = "") -> list:
    """
    Loads active tables from the table registry.
    Supports comma-separated filter terms: fully-qualified names or table names.
    """
    df = spark.table(TABLE_REGISTRY).filter(F.col("is_active") == True)

    if table_filter:
        terms = [t.strip().lower() for t in table_filter.split(",") if t.strip()]
        if terms:
            cond = (
                F.lower(F.col("fully_qualified_name")).isin(terms)
                | F.lower(F.element_at(F.split(F.col("fully_qualified_name"), "\\."), -1)).isin(terms)
            )
            df = df.filter(cond)

    tables = df.collect()
    if not tables:
        raise ValueError(f"No active tables matched table_filter='{table_filter}'.")

    print(f"Active tables found: {len(tables)}")
    for t in tables:
        print(f"  -> {t['fully_qualified_name']}")
    return tables


def load_active_columns(fqn: str) -> list:
    """
    Returns active column names for a given table.
    """
    rows = (
        spark.table(COLUMN_REGISTRY)
             .filter(
                 (F.col("fully_qualified_name") == fqn) &
                 (F.col("is_active") == True)
             )
             .collect()
    )
    return [r["column_name"] for r in rows]


def load_column_validations(fqn: str) -> dict:
    """
    Returns a dict: column_name -> set of validation_names
    Only active validations.
    """
    rows = (
        spark.table(COLUMN_VALIDATION_REGISTRY)
             .filter(
                 (F.col("fully_qualified_name") == fqn) &
                 (F.col("is_active") == True)
             )
             .collect()
    )
    result = {}
    for r in rows:
        col = r["column_name"]
        val = r["validation_name"]
        if col not in result:
            result[col] = set()
        result[col].add(val)
    return result


def load_table_validations(fqn: str) -> set:
    """
    Returns a set of active table-level validation names.
    """
    rows = (
        spark.table(TABLE_VALIDATION_REGISTRY)
             .filter(
                 (F.col("fully_qualified_name") == fqn) &
                 (F.col("is_active") == True)
             )
             .collect()
    )
    return {r["validation_name"] for r in rows}


active_tables = load_active_tables(TABLE_FILTER)
print(f"\nConfiguration loaded successfully.")

## Step 2: Baseline Computation Helper Functions

In [0]:
def get_baseline_window_dates(table_row) -> tuple:
    """
    Resolves runtime baseline window per table.
    Uses table-level baseline_lag_days from table registry when available.
    """
    global WINDOW_START_DATE, WINDOW_END_DATE, EFFECTIVE_DATE, BASELINE_WINDOW_DAYS

    if isinstance(table_row, dict):
        lag_raw = table_row.get("baseline_lag_days")
    else:
        lag_raw = table_row.get("baseline_lag_days") if hasattr(table_row, "get") else None
    lag_days = normalize_lag_days(lag_raw if lag_raw is not None else 1)

    # Custom parameters always win when both are provided.
    if BASELINE_START_RAW and BASELINE_END_RAW:
        start_dt = _safe_parse_date(BASELINE_START_RAW)
        end_dt = _safe_parse_date(BASELINE_END_RAW)
        if not start_dt or not end_dt or start_dt > end_dt:
            raise ValueError("Invalid baseline_start/baseline_end. Ensure baseline_start <= baseline_end.")
    else:
        start_dt, end_dt = compute_default_window(
            ANCHOR_DATE,
            lag_days,
            WINDOW_POLICY,
            _as_int_or_default(WINDOW_DAYS_RAW, 30),
        )

    WINDOW_START_DATE = start_dt
    WINDOW_END_DATE = end_dt
    EFFECTIVE_DATE = ANCHOR_DATE
    BASELINE_WINDOW_DAYS = (end_dt - start_dt).days + 1
    return WINDOW_START_DATE, WINDOW_END_DATE


def compute_days_with_data(
    fqn: str,
    business_date_col: str,
    start_date: date,
    end_date: date,
 ) -> int:
    """
    Distinct business dates with rows in the window,
    using the same date expression/scoping as the volume baseline.
    """
    df = load_scoped_table(fqn, business_date_col)
    if business_date_col and business_date_col in df.columns:
        return int(
            df.filter(
                (biz_date_expr(business_date_col) >= F.lit(start_date)) &
                (biz_date_expr(business_date_col) <= F.lit(end_date))
            )
            .select(biz_date_expr(business_date_col).alias("biz_date"))
            .where(F.col("biz_date").isNotNull())
            .distinct()
            .count()
        )
    # No business date column => one aggregate sample.
    return 1


def parse_custom_parameters(custom_params_str: str) -> dict:
    if not custom_params_str:
        return {}
    try:
        return json.loads(custom_params_str)
    except Exception:
        return {}


def build_baseline_record(
    fqn         : str,
    column_name : str,
    check_type  : str,
    metric_name : str,
    stats       : dict,
    window_days : int,
    baseline_detail = None
) -> dict:
    """
    Builds a single baseline record dictionary
    ready for writing to uc1_baseline_metrics_b.
    """
    sample_count  = int(stats.get("sample_count") or 0)
    is_sufficient = sample_count >= MIN_SAMPLE_COUNT

    mean_val = stats.get("mean_value")

    detail_payload = baseline_detail
    if detail_payload is None:
        detail_payload = stats.get("baseline_detail")

    if isinstance(detail_payload, (dict, list)):
        detail_json = json.dumps(detail_payload)
    elif detail_payload is None:
        detail_json = None
    else:
        detail_json = str(detail_payload)

    return {
        "baseline_id"          : str(uuid.uuid4()),
        "baseline_version_id"  : BASELINE_VERSION_ID,
        "fully_qualified_name" : fqn,
        "column_name"          : column_name,
        "check_type"           : check_type,
        "metric_name"          : metric_name,
        "baseline_date"        : BASELINE_DATE,
        "effective_date"       : EFFECTIVE_DATE,
        "window_start"         : WINDOW_START_DATE,
        "window_end"           : WINDOW_END_DATE,
        "run_mode"             : RUN_MODE,
        "job_run_id"           : JOB_RUN_ID or None,
        "baseline_window_days" : window_days,
        "expected_value"       : mean_val,
        "mean_value"           : mean_val,
        "stddev_value"         : stats.get("stddev_value"),
        "min_value"            : stats.get("min_value"),
        "max_value"            : stats.get("max_value"),
        "p25_value"            : stats.get("p25_value"),
        "p50_value"            : stats.get("p50_value"),
        "p75_value"            : stats.get("p75_value"),
        "p95_value"            : stats.get("p95_value"),
        "sample_count"         : sample_count,
        "is_sufficient_data"   : is_sufficient,
        "baseline_status"      : "VALID" if is_sufficient else "INSUFFICIENT_DATA",
        "baseline_detail"      : detail_json,
        "computed_timestamp"   : datetime.now(),
        "computed_by"          : COMPUTED_BY
    }


def safe_float(value) -> float:
    """Safely converts a value to float, returning None if not possible."""
    try:
        return float(value) if value is not None else None
    except (TypeError, ValueError):
        return None


print("Helper functions loaded.")

In [0]:
import re
from pyspark.sql.types import ArrayType, StructType

# Registry paths follow the Unity Catalog schema view: the element of an
# array is called `items`, e.g. stores.selected.items.id.
# Legacy paths with [] (stores.selected[].id) are still accepted.
ARRAY_ELEMENT = "items"


def normalize_nested_path(path: str) -> str:
    p = (path or "").replace("[]", f".{ARRAY_ELEMENT}")
    while ".." in p:
        p = p.replace("..", ".")
    return p.strip(".")


def is_nested_path(column_name: str) -> bool:
    return "." in (column_name or "") or "[]" in (column_name or "")


def resolve_nested_column(df, column_name: str):
    """
    Resolves registry column paths like touchpoint.id or stores.selected.items.id
    into a DataFrame plus a materialized alias column name.
    Arrays are exploded with explode_outer so nested array elements can be
    baselined like primitive columns.
    """
    if not column_name:
        return df, None
    if column_name in df.columns:
        return df, column_name

    # Schema-aware walk: `items` explodes only when the current type is an
    # array, so a struct field literally named `items` stays a field.
    segs = normalize_nested_path(column_name).split(".")
    root = segs[0]
    if root not in df.columns:
        return df, None
    cur = F.col(f"`{root}`")
    dt  = df.schema[root].dataType
    explode_idx = 0
    for seg in segs[1:]:
        if isinstance(dt, ArrayType) and seg == ARRAY_ELEMENT:
            explode_idx += 1
            tmp = f"__ds_exploded_{explode_idx}"
            df  = df.withColumn(tmp, F.explode_outer(cur))
            cur = F.col(tmp)
            dt  = dt.elementType
        elif isinstance(dt, StructType) and seg in dt.fieldNames():
            cur = cur.getField(seg)
            dt  = dt[seg].dataType
        else:
            print(f"      Field '{seg}' not found while resolving {column_name}.")
            return df, None

    alias = "__ds_" + re.sub(r"[^0-9A-Za-z_]", "_", normalize_nested_path(column_name))
    df = df.withColumn(alias, cur)
    return df, alias


# ── Business-date and row-scope helpers (keep in sync with 02_validation_agent) ──
# Date columns are often timestamp strings (2026-10-14T07:21:28.665).
# Filter and group on the DATE part so one baseline sample = one day.
SCD_END_COL = "__END_AT"    # NULL = current version of the record
ARRIVAL_COL = "__START_AT"  # when this version landed


def biz_date_expr(col_name: str):
    """Date part of a DATE / TIMESTAMP / date-like STRING column."""
    return F.expr(f"try_cast(`{col_name}` AS DATE)")


def load_scoped_table(fqn: str, business_date_col: str = ""):
    """
    Same row scope as the validation agent:
    - date column = __START_AT: all versions that landed each day
    - otherwise: current version only (__END_AT IS NULL)
    """
    df = spark.table(fqn)
    if SCD_END_COL in df.columns and business_date_col != ARRIVAL_COL:
        df = df.filter(F.col(SCD_END_COL).isNull())
    return df


print("Nested column helpers ready.")

## Step 3: Volume Anomaly Baseline

Computes daily row count statistics over the baseline window.
One baseline record per table.

In [0]:
def compute_volume_baseline(
    fqn                 : str,
    business_date_col   : str,
    start_date          : date,
    end_date            : date,
    window_days         : int
) -> dict:
    """
    Computes row count statistics per business date
    over the baseline window.

    Returns a stats dict or None on failure.
    """
    print(f"    [VOLUME_ANOMALY] Computing for {fqn}")
    print(f"      Window: {start_date} to {end_date}")

    try:
        df = load_scoped_table(fqn, business_date_col)

        if business_date_col and business_date_col in df.columns:
            daily_counts = (
                df.filter(
                    (biz_date_expr(business_date_col) >= F.lit(start_date)) &
                    (biz_date_expr(business_date_col) <= F.lit(end_date))
                )
                .groupBy(biz_date_expr(business_date_col).alias(business_date_col))
                .agg(F.count("*").alias("row_count"))
            )
        else:
            # No business date column: use total row count
            total = df.count()
            print(f"      No business date column. Using total row count: {total}")
            return {
                "mean_value"   : float(total),
                "stddev_value" : 0.0,
                "min_value"    : float(total),
                "max_value"    : float(total),
                "p25_value"    : float(total),
                "p50_value"    : float(total),
                "p75_value"    : float(total),
                "p95_value"    : float(total),
                "sample_count" : 1
            }

        stats_row = daily_counts.agg(
            F.mean("row_count")
             .alias("mean_value"),
            F.stddev("row_count")
             .alias("stddev_value"),
            F.min("row_count")
             .alias("min_value"),
            F.max("row_count")
             .alias("max_value"),
            F.percentile_approx("row_count", 0.25)
             .alias("p25_value"),
            F.percentile_approx("row_count", 0.50)
             .alias("p50_value"),
            F.percentile_approx("row_count", 0.75)
             .alias("p75_value"),
            F.percentile_approx("row_count", 0.95)
             .alias("p95_value"),
            F.count("*")
             .alias("sample_count")
        ).collect()[0]

        result = {
            "mean_value"   : safe_float(stats_row["mean_value"]),
            "stddev_value" : safe_float(stats_row["stddev_value"]),
            "min_value"    : safe_float(stats_row["min_value"]),
            "max_value"    : safe_float(stats_row["max_value"]),
            "p25_value"    : safe_float(stats_row["p25_value"]),
            "p50_value"    : safe_float(stats_row["p50_value"]),
            "p75_value"    : safe_float(stats_row["p75_value"]),
            "p95_value"    : safe_float(stats_row["p95_value"]),
            "sample_count" : int(stats_row["sample_count"] or 0)
        }

        print(f"      Mean daily rows : {result['mean_value']:.1f}")
        print(f"      StdDev          : {result['stddev_value']:.1f}")
        print(f"      Min / Max       : {result['min_value']:.0f} / {result['max_value']:.0f}")
        print(f"      Sample days     : {result['sample_count']}")

        return result

    except Exception as e:
        print(f"      ERROR: {str(e)}")
        return None


print("Volume baseline function loaded.")

## Step 4: Null Spike Baseline

Computes daily null rate statistics for each monitored column.
One baseline record per column per table.

In [0]:
def compute_null_rate_baseline(
    fqn               : str,
    column_name       : str,
    business_date_col : str,
    start_date        : date,
    end_date          : date,
    window_days       : int
) -> dict:
    """
    Computes daily null rate statistics for a specific column.

    For columns marked with invert=true in custom_parameters
    (like _rescued_data), we compute the non-null rate instead.
    """
    print(f"    [NULL_SPIKE] Computing for {fqn} → {column_name}")

    try:
        df = load_scoped_table(fqn, business_date_col)
        df, value_col = resolve_nested_column(df, column_name)

        if not value_col:
            print(f"      Column {column_name} not found. Skipping.")
            return None

        if business_date_col and business_date_col in df.columns:
            df_filtered = df.filter(
                (biz_date_expr(business_date_col) >= F.lit(start_date)) &
                (biz_date_expr(business_date_col) <= F.lit(end_date))
            )

            daily_null_rates = (
                df_filtered
                .groupBy(biz_date_expr(business_date_col).alias(business_date_col))
                .agg(
                    (
                        F.sum(F.col(value_col).isNull().cast("int")) /
                        F.count("*")
                    ).alias("null_rate"),
                    F.count("*").alias("total_records")
                )
            )
        else:
            total      = df.count()
            null_count = df.filter(F.col(value_col).isNull()).count()
            null_rate  = null_count / total if total > 0 else 0.0
            return {
                "mean_value"   : null_rate,
                "stddev_value" : 0.0,
                "min_value"    : null_rate,
                "max_value"    : null_rate,
                "p25_value"    : null_rate,
                "p50_value"    : null_rate,
                "p75_value"    : null_rate,
                "p95_value"    : null_rate,
                "sample_count" : 1
            }

        stats_row = daily_null_rates.agg(
            F.mean("null_rate")
             .alias("mean_value"),
            F.stddev("null_rate")
             .alias("stddev_value"),
            F.min("null_rate")
             .alias("min_value"),
            F.max("null_rate")
             .alias("max_value"),
            F.percentile_approx("null_rate", 0.25)
             .alias("p25_value"),
            F.percentile_approx("null_rate", 0.50)
             .alias("p50_value"),
            F.percentile_approx("null_rate", 0.75)
             .alias("p75_value"),
            F.percentile_approx("null_rate", 0.95)
             .alias("p95_value"),
            F.count("*")
             .alias("sample_count")
        ).collect()[0]

        result = {
            "mean_value"   : safe_float(stats_row["mean_value"]),
            "stddev_value" : safe_float(stats_row["stddev_value"]),
            "min_value"    : safe_float(stats_row["min_value"]),
            "max_value"    : safe_float(stats_row["max_value"]),
            "p25_value"    : safe_float(stats_row["p25_value"]),
            "p50_value"    : safe_float(stats_row["p50_value"]),
            "p75_value"    : safe_float(stats_row["p75_value"]),
            "p95_value"    : safe_float(stats_row["p95_value"]),
            "sample_count" : int(stats_row["sample_count"] or 0)
        }

        print(f"      Mean null rate  : {(result['mean_value'] or 0)*100:.4f}%")
        print(f"      Max null rate   : {(result['max_value'] or 0)*100:.4f}%")
        print(f"      Sample days     : {result['sample_count']}")

        return result

    except Exception as e:
        print(f"      ERROR: {str(e)}")
        return None


print("Null spike baseline function loaded.")

## Step 5: Categorical Domain Baseline

Captures the set of distinct values seen in clean historical data.
Used to detect new or invalid values in subsequent loads.

In [0]:
def compute_categorical_baseline(
    fqn               : str,
    column_name       : str,
    business_date_col : str,
    start_date        : date,
    end_date          : date,
    window_days       : int
) -> dict:
    """
    Computes the distinct value count and value distribution
    for a categorical column over the baseline window.

    The distinct count serves as the baseline for CARDINALITY_DRIFT.
    For CATEGORICAL_DOMAIN, we also learn a baseline allow-list
    from the most frequent values in the baseline window.
    """
    print(f"    [CATEGORICAL/CARDINALITY] Computing for {fqn} -> {column_name}")

    try:
        df = load_scoped_table(fqn, business_date_col)
        df, value_col = resolve_nested_column(df, column_name)

        if not value_col:
            print(f"      Column {column_name} not found. Skipping.")
            return None

        if business_date_col and business_date_col in df.columns:
            df_filtered = df.filter(
                (biz_date_expr(business_date_col) >= F.lit(start_date)) &
                (biz_date_expr(business_date_col) <= F.lit(end_date))
            )
        else:
            df_filtered = df

        # Distinct count per day
        if business_date_col and business_date_col in df.columns:
            daily_cardinality = (
                df_filtered
                .groupBy(biz_date_expr(business_date_col).alias(business_date_col))
                .agg(
                    F.countDistinct(value_col).alias("distinct_count")
                )
            )

            stats_row = daily_cardinality.agg(
                F.mean("distinct_count")
                 .alias("mean_value"),
                F.stddev("distinct_count")
                 .alias("stddev_value"),
                F.min("distinct_count")
                 .alias("min_value"),
                F.max("distinct_count")
                 .alias("max_value"),
                F.percentile_approx("distinct_count", 0.25)
                 .alias("p25_value"),
                F.percentile_approx("distinct_count", 0.50)
                 .alias("p50_value"),
                F.percentile_approx("distinct_count", 0.75)
                 .alias("p75_value"),
                F.percentile_approx("distinct_count", 0.95)
                 .alias("p95_value"),
                F.count("*")
                 .alias("sample_count")
            ).collect()[0]
        else:
            distinct = df_filtered.select(value_col).distinct().count()
            stats_row = {
                "mean_value"   : float(distinct),
                "stddev_value" : 0.0,
                "min_value"    : float(distinct),
                "max_value"    : float(distinct),
                "p25_value"    : float(distinct),
                "p50_value"    : float(distinct),
                "p75_value"    : float(distinct),
                "p95_value"    : float(distinct),
                "sample_count" : 1
            }

        # Learn baseline domain from top frequent non-null values.
        top_values_rows = (
            df_filtered
            .filter(F.col(value_col).isNotNull())
            .groupBy(F.col(value_col).cast("string").alias("value"))
            .count()
            .orderBy(F.col("count").desc())
            .limit(100)
            .collect()
        )
        valid_values = [r["value"] for r in top_values_rows if r["value"] is not None]
        top_values = [{"value": r["value"], "count": int(r["count"])} for r in top_values_rows]

        result = {
            "mean_value"   : safe_float(stats_row["mean_value"]),
            "stddev_value" : safe_float(stats_row["stddev_value"]),
            "min_value"    : safe_float(stats_row["min_value"]),
            "max_value"    : safe_float(stats_row["max_value"]),
            "p25_value"    : safe_float(stats_row["p25_value"]),
            "p50_value"    : safe_float(stats_row["p50_value"]),
            "p75_value"    : safe_float(stats_row["p75_value"]),
            "p95_value"    : safe_float(stats_row["p95_value"]),
            "sample_count" : int(stats_row["sample_count"] or 0),
            "baseline_detail": {
                "valid_values": valid_values,
                "top_values": top_values,
            },
        }

        print(f"      Mean distinct   : {result['mean_value']:.1f}")
        print(f"      Min / Max       : {result['min_value']:.0f} / {result['max_value']:.0f}")
        print(f"      Sample days     : {result['sample_count']}")
        print(f"      Learned domain  : {len(valid_values)} values")

        return result

    except Exception as e:
        print(f"      ERROR: {str(e)}")
        return None


print("Categorical baseline function loaded.")

## Step 6: Numeric Distribution Baseline

Computes value distribution statistics for numeric columns.
Used for NUMERIC_RANGE and DISTRIBUTION_DRIFT checks.

In [0]:
def compute_numeric_baseline(
    fqn               : str,
    column_name       : str,
    business_date_col : str,
    start_date        : date,
    end_date          : date,
    window_days       : int
) -> dict:
    """
    Computes value distribution statistics for a numeric column.
    Operates on all non-null values within the baseline window.
    """
    print(f"    [NUMERIC_RANGE] Computing for {fqn} → {column_name}")

    try:
        df = load_scoped_table(fqn, business_date_col)
        df, value_col = resolve_nested_column(df, column_name)

        if not value_col:
            print(f"      Column {column_name} not found. Skipping.")
            return None

        if business_date_col and business_date_col in df.columns:
            df_filtered = df.filter(
                (biz_date_expr(business_date_col) >= F.lit(start_date)) &
                (biz_date_expr(business_date_col) <= F.lit(end_date))
            )
        else:
            df_filtered = df

        # Only non-null values
        df_col = (
            df_filtered
            .filter(F.col(value_col).isNotNull())
            .select(F.col(value_col).alias("value_col"))
        )

        total_records = df_col.count()
        if total_records == 0:
            print(f"      No non-null records found for {column_name}. Skipping.")
            return None

        stats_row = df_col.agg(
            F.mean("value_col")
             .alias("mean_value"),
            F.stddev("value_col")
             .alias("stddev_value"),
            F.min("value_col")
             .alias("min_value"),
            F.max("value_col")
             .alias("max_value"),
            F.percentile_approx("value_col", 0.25)
             .alias("p25_value"),
            F.percentile_approx("value_col", 0.50)
             .alias("p50_value"),
            F.percentile_approx("value_col", 0.75)
             .alias("p75_value"),
            F.percentile_approx("value_col", 0.95)
             .alias("p95_value"),
            F.count("*")
             .alias("sample_count")
        ).collect()[0]

        result = {
            "mean_value"   : safe_float(stats_row["mean_value"]),
            "stddev_value" : safe_float(stats_row["stddev_value"]),
            "min_value"    : safe_float(stats_row["min_value"]),
            "max_value"    : safe_float(stats_row["max_value"]),
            "p25_value"    : safe_float(stats_row["p25_value"]),
            "p50_value"    : safe_float(stats_row["p50_value"]),
            "p75_value"    : safe_float(stats_row["p75_value"]),
            "p95_value"    : safe_float(stats_row["p95_value"]),
            "sample_count" : int(stats_row["sample_count"] or 0)
        }

        print(f"      Mean value      : {result['mean_value']:.4f}")
        print(f"      StdDev          : {result['stddev_value']:.4f}")
        print(f"      Min / Max       : {result['min_value']:.4f} / {result['max_value']:.4f}")
        print(f"      Sample records  : {result['sample_count']}")

        return result

    except Exception as e:
        print(f"      ERROR: {str(e)}")
        return None


print("Numeric baseline function loaded.")

In [ ]:
def load_table_validation_configs(fqn: str) -> dict:
    rows = (
        spark.table(TABLE_VALIDATION_REGISTRY)
             .filter(
                 (F.col("fully_qualified_name") == fqn) &
                 (F.col("is_active") == True)
             )
             .collect()
    )
    result = {}
    for row in rows:
        raw_name = str(row["validation_name"])
        check_name = TABLE_CHECK_NAME_MAP.get(raw_name, raw_name)
        if check_name in result:
            continue
        cfg = row.asDict()
        cfg["raw_validation_name"] = raw_name
        cfg["validation_name"] = check_name
        cfg["custom_parameters_parsed"] = parse_custom_parameters(cfg.get("custom_parameters"))
        result[check_name] = cfg
    return result


def get_business_key_columns(fqn: str) -> list:
    rows = (
        spark.table(COLUMN_REGISTRY)
             .filter(
                 (F.col("fully_qualified_name") == fqn) &
                 (F.col("is_active") == True) &
                 (F.col("is_business_key") == True)
             )
             .collect()
    )
    return [str(r["column_name"]) for r in rows]


def _extract_daily_metric_stats(daily_df, metric_col: str) -> dict:
    row = daily_df.agg(
        F.mean(metric_col).alias("mean_value"),
        F.stddev(metric_col).alias("stddev_value"),
        F.min(metric_col).alias("min_value"),
        F.max(metric_col).alias("max_value"),
        F.percentile_approx(metric_col, 0.25).alias("p25_value"),
        F.percentile_approx(metric_col, 0.50).alias("p50_value"),
        F.percentile_approx(metric_col, 0.75).alias("p75_value"),
        F.percentile_approx(metric_col, 0.95).alias("p95_value"),
        F.count("*").alias("sample_count"),
    ).collect()[0]
    return {
        "mean_value": safe_float(row["mean_value"]),
        "stddev_value": safe_float(row["stddev_value"]),
        "min_value": safe_float(row["min_value"]),
        "max_value": safe_float(row["max_value"]),
        "p25_value": safe_float(row["p25_value"]),
        "p50_value": safe_float(row["p50_value"]),
        "p75_value": safe_float(row["p75_value"]),
        "p95_value": safe_float(row["p95_value"]),
        "sample_count": int(row["sample_count"] or 0),
    }


def compute_partition_freshness_baseline(
    fqn: str,
    biz_date_col: str,
    start_date: date,
    end_date: date,
) -> dict:
    df = load_scoped_table(fqn, biz_date_col)
    if not biz_date_col or biz_date_col not in df.columns:
        return {
            "mean_value": 0.0,
            "stddev_value": 0.0,
            "min_value": 0.0,
            "max_value": 0.0,
            "p25_value": 0.0,
            "p50_value": 0.0,
            "p75_value": 0.0,
            "p95_value": 0.0,
            "sample_count": 1,
            "baseline_detail": {"reason": "no_business_date_column"},
        }

    df_w = df.filter(
        (biz_date_expr(biz_date_col) >= F.lit(start_date)) &
        (biz_date_expr(biz_date_col) <= F.lit(end_date))
    )

    if ARRIVAL_COL in df_w.columns:
        daily = (
            df_w.groupBy(biz_date_expr(biz_date_col).alias("biz_date"))
                .agg(F.max(F.col(ARRIVAL_COL)).alias("max_arrival_ts"))
                .withColumn(
                    "expected_ts",
                    F.to_timestamp(F.concat(F.col("biz_date").cast("string"), F.lit(" 00:00:00")))
                )
                .withColumn(
                    "lag_minutes",
                    F.greatest(
                        (F.unix_timestamp(F.col("max_arrival_ts")) - F.unix_timestamp(F.col("expected_ts"))) / F.lit(60.0),
                        F.lit(0.0),
                    ),
                )
                .select("biz_date", "lag_minutes")
        )
    else:
        daily = (
            df_w.groupBy(biz_date_expr(biz_date_col).alias("biz_date"))
                .agg(F.count("*").alias("_rows"))
                .withColumn("lag_minutes", F.lit(0.0))
                .select("biz_date", "lag_minutes")
        )

    stats = _extract_daily_metric_stats(daily, "lag_minutes")
    stats["baseline_detail"] = {
        "arrival_column_used": ARRIVAL_COL if ARRIVAL_COL in df_w.columns else None,
        "business_date_column": biz_date_col,
    }
    return stats


def compute_duplicate_ingestion_baseline(
    fqn: str,
    biz_date_col: str,
    start_date: date,
    end_date: date,
    key_columns: list,
) -> dict:
    if not key_columns:
        return {
            "mean_value": 0.0,
            "stddev_value": 0.0,
            "min_value": 0.0,
            "max_value": 0.0,
            "p25_value": 0.0,
            "p50_value": 0.0,
            "p75_value": 0.0,
            "p95_value": 0.0,
            "sample_count": 1,
            "baseline_detail": {"reason": "no_business_key_columns"},
        }

    df = load_scoped_table(fqn, biz_date_col)
    present_keys = [k for k in key_columns if k in df.columns]
    if not present_keys:
        return {
            "mean_value": 0.0,
            "stddev_value": 0.0,
            "min_value": 0.0,
            "max_value": 0.0,
            "p25_value": 0.0,
            "p50_value": 0.0,
            "p75_value": 0.0,
            "p95_value": 0.0,
            "sample_count": 1,
            "baseline_detail": {"reason": "key_columns_not_found", "requested_key_columns": key_columns},
        }

    if biz_date_col and biz_date_col in df.columns:
        df_w = df.filter(
            (biz_date_expr(biz_date_col) >= F.lit(start_date)) &
            (biz_date_expr(biz_date_col) <= F.lit(end_date))
        )

        daily = (
            df_w.groupBy(biz_date_expr(biz_date_col).alias("biz_date"))
                .agg(
                    F.count("*").alias("row_count"),
                    F.countDistinct(F.struct(*[F.col(c) for c in present_keys])).alias("distinct_key_count"),
                )
                .withColumn(
                    "duplicate_pct",
                    F.when(
                        F.col("row_count") > 0,
                        (F.col("row_count") - F.col("distinct_key_count")) / F.col("row_count") * F.lit(100.0),
                    ).otherwise(F.lit(0.0)),
                )
                .select("biz_date", "duplicate_pct")
        )
        stats = _extract_daily_metric_stats(daily, "duplicate_pct")
    else:
        row_count = df.count()
        distinct_count = df.select(F.struct(*[F.col(c) for c in present_keys]).alias("k")).distinct().count()
        dup_pct = ((row_count - distinct_count) / row_count * 100.0) if row_count > 0 else 0.0
        stats = {
            "mean_value": dup_pct,
            "stddev_value": 0.0,
            "min_value": dup_pct,
            "max_value": dup_pct,
            "p25_value": dup_pct,
            "p50_value": dup_pct,
            "p75_value": dup_pct,
            "p95_value": dup_pct,
            "sample_count": 1,
        }

    stats["baseline_detail"] = {
        "business_key_columns": present_keys,
        "requested_business_key_columns": key_columns,
    }
    return stats


print("Extended table baseline helpers loaded.")

## Step 7: Write Baseline Records to uc1_baseline_metrics_b

In [0]:
def write_baseline_records(records: list) -> int:
    """
    Writes baseline records to uc1_baseline_metrics_b.
    Uses MERGE to handle reruns gracefully.
    Existing baselines for the same table/column/check/date/version
    are updated rather than duplicated.
    """
    if not records:
        print("  No records to write.")
        return 0

    assert_metadata_target(BASELINE_METRICS)

    # Convert to tuples in schema column order
    column_order = [field.name for field in BASELINE_SCHEMA.fields]

    rows = [
        tuple(r.get(col) for col in column_order)
        for r in records
    ]

    df = spark.createDataFrame(rows, schema=BASELINE_SCHEMA)

    df.createOrReplaceTempView("baseline_staging")

    spark.sql(f"""
        MERGE INTO {BASELINE_METRICS} AS target
        USING baseline_staging AS source
        ON  target.fully_qualified_name = source.fully_qualified_name
        AND target.column_name          = source.column_name
        AND target.check_type           = source.check_type
        AND target.metric_name          = source.metric_name
        AND target.baseline_date        = source.baseline_date
        AND target.baseline_version_id  = source.baseline_version_id
        WHEN MATCHED THEN
            UPDATE SET *
        WHEN NOT MATCHED THEN
            INSERT *
    """)

    print(f"  Written {len(records)} baseline records.")
    return len(records)


print("Write function loaded.")

## Step 8: Run Baseline Computation for All Active Tables

In [0]:
all_baseline_records = []
tables_processed     = 0
tables_skipped       = 0
tables_failed        = 0

for table_row in active_tables:
    fqn               = table_row["fully_qualified_name"]
    biz_date_col      = table_row["business_date_column"] or ""

    print(f"\n{'='*60}")
    print(f"Processing: {fqn}")
    print(f"Business date column: {biz_date_col or 'None'}")
    print(f"{'='*60}")

    try:
        # -- Load configuration for this table ---------------------------------
        active_columns = load_active_columns(fqn)
        col_validations = load_column_validations(fqn)
        table_validation_configs = load_table_validation_configs(fqn)
        table_validations = set(table_validation_configs.keys())

        print(f"Active columns     : {len(active_columns)}")
        print(f"Table validations  : {sorted(table_validations)}")
        print(f"Column validations : {len(col_validations)} columns configured")

        if not table_validations and not col_validations:
            print("No validations configured. Skipping.")
            tables_skipped += 1
            continue

        # -- Compute baseline window dates -------------------------------------
        start_date, end_date = get_baseline_window_dates(table_row)
        print(f"Baseline window    : {start_date} to {end_date}")
        print()

        days_with_data = compute_days_with_data(fqn, biz_date_col, start_date, end_date)
        has_min_days = days_with_data >= MIN_BASELINE_DAYS
        print(f"Days with data     : {days_with_data}")

        table_records = []

        # -- Table-Level Baselines ---------------------------------------------

        if "VOLUME_ANOMALY" in table_validations:
            stats = compute_volume_baseline(
                fqn, biz_date_col, start_date, end_date, BASELINE_WINDOW_DAYS
            )
            if stats:
                table_records.append(
                    build_baseline_record(
                        fqn         = fqn,
                        column_name = "__table__",
                        check_type  = "VOLUME_ANOMALY",
                        metric_name = "daily_row_count",
                        stats       = stats,
                        window_days = BASELINE_WINDOW_DAYS
                    )
                )

        if "SCHEMA_CONFORMANCE" in table_validations:
            try:
                schema_obj = spark.table(fqn).schema
                schema_fields = schema_obj.fields
                col_count = len(schema_fields)
                schema_snapshot = [
                    {"name": f.name, "type": f.dataType.simpleString()}
                    for f in schema_fields
                ]

                stats = {
                    "mean_value"   : float(col_count),
                    "stddev_value" : 0.0,
                    "min_value"    : float(col_count),
                    "max_value"    : float(col_count),
                    "p25_value"    : float(col_count),
                    "p50_value"    : float(col_count),
                    "p75_value"    : float(col_count),
                    "p95_value"    : float(col_count),
                    "sample_count" : 1,
                }

                table_records.append(
                    build_baseline_record(
                        fqn         = fqn,
                        column_name = "__table__",
                        check_type  = "SCHEMA_CONFORMANCE",
                        metric_name = "column_count",
                        stats       = stats,
                        window_days = BASELINE_WINDOW_DAYS,
                        baseline_detail = {
                            "column_count": col_count,
                            "schema_snapshot": schema_snapshot,
                        },
                    )
                )
                print(f"    [SCHEMA_CONFORMANCE] Column count baseline: {col_count}")
            except Exception as e:
                print(f"    [SCHEMA_CONFORMANCE] ERROR: {str(e)}")

        if "MALFORMED_RECORDS" in table_validations:
            stats = compute_null_rate_baseline(
                fqn, "_rescued_data", biz_date_col,
                start_date, end_date, BASELINE_WINDOW_DAYS
            )
            if stats:
                table_records.append(
                    build_baseline_record(
                        fqn         = fqn,
                        column_name = "__table__",
                        check_type  = "MALFORMED_RECORDS",
                        metric_name = "rescued_data_rate",
                        stats       = stats,
                        window_days = BASELINE_WINDOW_DAYS,
                        baseline_detail = {"indicator_column": "_rescued_data"},
                    )
                )

        if "PARTITION_FRESHNESS" in table_validations:
            tv_cfg = table_validation_configs.get("PARTITION_FRESHNESS", {})
            params = tv_cfg.get("custom_parameters_parsed") or {}
            stats = compute_partition_freshness_baseline(
                fqn, biz_date_col, start_date, end_date
            )
            if stats:
                detail = dict(stats.get("baseline_detail") or {})
                detail["expected_latency_minutes"] = params.get("expected_latency_minutes")
                detail["expected_hour"] = params.get("expected_hour")
                table_records.append(
                    build_baseline_record(
                        fqn         = fqn,
                        column_name = "__table__",
                        check_type  = "PARTITION_FRESHNESS",
                        metric_name = "freshness_lag_minutes",
                        stats       = stats,
                        window_days = BASELINE_WINDOW_DAYS,
                        baseline_detail = detail,
                    )
                )

        if "DUPLICATE_INGESTION" in table_validations:
            tv_cfg = table_validation_configs.get("DUPLICATE_INGESTION", {})
            params = tv_cfg.get("custom_parameters_parsed") or {}
            key_cols = params.get("business_key_columns") or get_business_key_columns(fqn)
            stats = compute_duplicate_ingestion_baseline(
                fqn, biz_date_col, start_date, end_date, key_cols
            )
            if stats:
                table_records.append(
                    build_baseline_record(
                        fqn         = fqn,
                        column_name = "__table__",
                        check_type  = "DUPLICATE_INGESTION",
                        metric_name = "duplicate_percentage",
                        stats       = stats,
                        window_days = BASELINE_WINDOW_DAYS,
                    )
                )

        # -- Column-Level Baselines --------------------------------------------

        for col_name in active_columns:
            checks_for_col = col_validations.get(col_name, set())

            if not checks_for_col:
                continue

            print(f"\n  Column: {col_name}")
            print(f"  Checks: {checks_for_col}")

            if "NULL_SPIKE" in checks_for_col:
                stats = compute_null_rate_baseline(
                    fqn, col_name, biz_date_col,
                    start_date, end_date, BASELINE_WINDOW_DAYS
                )
                if stats:
                    table_records.append(
                        build_baseline_record(
                            fqn         = fqn,
                            column_name = col_name,
                            check_type  = "NULL_SPIKE",
                            metric_name = "null_rate",
                            stats       = stats,
                            window_days = BASELINE_WINDOW_DAYS
                        )
                    )

            if "NUMERIC_RANGE" in checks_for_col:
                stats = compute_numeric_baseline(
                    fqn, col_name, biz_date_col,
                    start_date, end_date, BASELINE_WINDOW_DAYS
                )
                if stats:
                    table_records.append(
                        build_baseline_record(
                            fqn         = fqn,
                            column_name = col_name,
                            check_type  = "NUMERIC_RANGE",
                            metric_name = "value_distribution",
                            stats       = stats,
                            window_days = BASELINE_WINDOW_DAYS
                        )
                    )

            if "CATEGORICAL_DOMAIN" in checks_for_col or \
               "CARDINALITY_DRIFT"  in checks_for_col:
                stats = compute_categorical_baseline(
                    fqn, col_name, biz_date_col,
                    start_date, end_date, BASELINE_WINDOW_DAYS
                )
                if stats:
                    if "CATEGORICAL_DOMAIN" in checks_for_col:
                        table_records.append(
                            build_baseline_record(
                                fqn         = fqn,
                                column_name = col_name,
                                check_type  = "CATEGORICAL_DOMAIN",
                                metric_name = "distinct_value_count",
                                stats       = stats,
                                window_days = BASELINE_WINDOW_DAYS,
                            )
                        )
                    if "CARDINALITY_DRIFT" in checks_for_col:
                        table_records.append(
                            build_baseline_record(
                                fqn         = fqn,
                                column_name = col_name,
                                check_type  = "CARDINALITY_DRIFT",
                                metric_name = "distinct_value_count",
                                stats       = stats,
                                window_days = BASELINE_WINDOW_DAYS,
                            )
                        )

            if "CROSS_COLUMN_CONSISTENCY" in checks_for_col:
                stats = {
                    "mean_value"   : 0.0,
                    "stddev_value" : 0.0,
                    "min_value"    : 0.0,
                    "max_value"    : 0.0,
                    "p25_value"    : 0.0,
                    "p50_value"    : 0.0,
                    "p75_value"    : 0.0,
                    "p95_value"    : 0.0,
                    "sample_count" : BASELINE_WINDOW_DAYS,
                }
                table_records.append(
                    build_baseline_record(
                        fqn         = fqn,
                        column_name = col_name,
                        check_type  = "CROSS_COLUMN_CONSISTENCY",
                        metric_name = "violation_rate",
                        stats       = stats,
                        window_days = BASELINE_WINDOW_DAYS,
                    )
                )
                print(f"    [CROSS_COLUMN_CONSISTENCY] Baseline: 0% violation rate")

        if not has_min_days and table_records:
            print(f"INSUFFICIENT_DATA: {fqn} {days_with_data} days")
            for rec in table_records:
                rec["sample_count"] = int(days_with_data)
                rec["is_sufficient_data"] = False
                rec["baseline_status"] = "INSUFFICIENT_DATA"

        all_baseline_records.extend(table_records)
        tables_processed += 1
        print(f"\n  Completed. Running total: {len(all_baseline_records)} records.")

    except Exception as e:
        print(f"\n  FAILED: {str(e)}")
        tables_failed += 1

print(f"\n{'='*60}")
print("Loop complete.")
print(f"  Tables processed : {tables_processed}")
print(f"  Tables skipped   : {tables_skipped}")
print(f"  Tables failed    : {tables_failed}")
print(f"  Records to write : {len(all_baseline_records)}")
print(f"{'='*60}")

## Step 9: Write All Baseline Records

In [0]:
print("Writing baseline records to uc1_baseline_metrics_b...")
records_written = write_baseline_records(all_baseline_records)
print(f"Write complete. {records_written} records written.")

## Step 10: Verify Baseline Results

In [0]:
print("=" * 60)
print("BASELINE VERIFICATION")
print("=" * 60)

# Summary by table and check type
print("\n1. Baselines by table and check type:\n")
spark.sql(f"""
    SELECT
        fully_qualified_name,
        check_type,
        column_name,
        metric_name,
        ROUND(mean_value, 4)    AS mean_value,
        ROUND(stddev_value, 4)  AS stddev_value,
        ROUND(min_value, 4)     AS min_value,
        ROUND(max_value, 4)     AS max_value,
        sample_count,
        is_sufficient_data,
        baseline_status
    FROM {BASELINE_METRICS}
    WHERE baseline_date = '{BASELINE_DATE}'
    ORDER BY
        fully_qualified_name,
        check_type,
        column_name
""").show(100, truncate=False)

# Volume anomaly baselines specifically
print("\n2. Volume anomaly baselines:\n")
spark.sql(f"""
    SELECT
        fully_qualified_name,
        ROUND(mean_value, 1)    AS mean_daily_rows,
        ROUND(stddev_value, 1)  AS stddev,
        ROUND(min_value, 0)     AS min_rows,
        ROUND(max_value, 0)     AS max_rows,
        sample_count,
        baseline_status
    FROM {BASELINE_METRICS}
    WHERE baseline_date = '{BASELINE_DATE}'
      AND check_type = 'VOLUME_ANOMALY'
""").show(truncate=False)

# Null rate baselines
print("\n3. Null rate baselines (NULL_SPIKE):\n")
spark.sql(f"""
    SELECT
        fully_qualified_name,
        column_name,
        ROUND(mean_value * 100, 4)  AS mean_null_pct,
        ROUND(max_value * 100, 4)   AS max_null_pct,
        sample_count,
        baseline_status
    FROM {BASELINE_METRICS}
    WHERE baseline_date = '{BASELINE_DATE}'
      AND check_type = 'NULL_SPIKE'
    ORDER BY fully_qualified_name, column_name
""").show(truncate=False)

# Cardinality baselines
print("\n4. Cardinality baselines:\n")
spark.sql(f"""
    SELECT
        fully_qualified_name,
        column_name,
        check_type,
        ROUND(mean_value, 1)    AS mean_distinct_count,
        ROUND(min_value, 0)     AS min_distinct,
        ROUND(max_value, 0)     AS max_distinct,
        sample_count,
        baseline_status
    FROM {BASELINE_METRICS}
    WHERE baseline_date = '{BASELINE_DATE}'
      AND check_type IN ('CATEGORICAL_DOMAIN', 'CARDINALITY_DRIFT')
    ORDER BY fully_qualified_name, column_name
""").show(truncate=False)

# Overall summary
print("\n5. Overall baseline summary:\n")
spark.sql(f"""
    SELECT
        fully_qualified_name,
        COUNT(*)                                                AS total_baselines,
        SUM(CASE WHEN is_sufficient_data THEN 1 ELSE 0 END)    AS sufficient,
        SUM(CASE WHEN NOT is_sufficient_data THEN 1 ELSE 0 END) AS insufficient,
        COUNT(DISTINCT check_type)                             AS distinct_checks
    FROM {BASELINE_METRICS}
    WHERE baseline_date = '{BASELINE_DATE}'
    GROUP BY fully_qualified_name
    ORDER BY fully_qualified_name
""").show(truncate=False)

## Summary

In [0]:
print("\n" + "=" * 60)
print("BASELINE COMPUTATION COMPLETE")
print("=" * 60)
print(f"Tables processed         : {tables_processed}")
print(f"Tables skipped           : {tables_skipped}")
print(f"Tables failed            : {tables_failed}")
print(f"Baseline records written : {records_written}")
print(f"Baseline date            : {BASELINE_DATE}")
print(f"Baseline window          : {BASELINE_WINDOW_DAYS} days")
print(f"Completed at             : {datetime.now()}")
print("=" * 60)

if tables_failed > 0:
    print("\nWARNING: Some tables failed.")
    print("Check the output above for error details.")

if records_written == 0:
    print("\nACTION REQUIRED: No baselines were computed.")
    print("Ensure tables are registered and validations are configured.")
else:
    print("\nBaselines are ready.")
    print("You can now run the Agent 1 validation notebook.")